# Zimbabwe MICS 2019: Children's Reading Opportunity and Access to Books

This notebook presents a cleaned, reproducible secondary analysis of the **Zimbabwe Multiple Indicator Cluster Survey (MICS) 2019** 5–17-year-old dataset.

## Research focus
The analysis examines children's access to reading materials and selected indicators of reading opportunity and engagement.

**Focal variables**
- `PR3` — number of children's or picture books
- `FL6A` — child reads books
- `FL6B` — someone reads books to the child
- `FL10` — child likes reading stories

**Derived variables**
- `child_books_n` — valid PR3 values from 0–10
- `has_child_book` — no children's/picture books vs at least one

> **Important:** Results are unweighted descriptive statistics based on valid responses in the analysed sample. Survey weights and the MICS complex survey design are not applied, so these percentages are **not national prevalence estimates for Zimbabwe**.


## 1. Setup and data access

The original MICS microdata are **not redistributed with this repository**. To reproduce the analysis, obtain authorised Zimbabwe MICS 2019 microdata separately.

The code below follows the folder structure of the original MICS download used in the working analysis. If your downloaded files are stored differently, update `zip_path` and/or `fs_path`.


In [ ]:
# Install once in Google Colab if needed
%pip install -q pyreadstat


In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyreadstat


In [ ]:
# Path to the authorised Zimbabwe MICS 2019 download
zip_path = "/content/ZWE_2019_MICS6_v01_M.zip"
extract_path = "/content/zimbabwe_mics_2019"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_path)

inner_zip = os.path.join(
    extract_path,
    "ZWE_2019_MICS6_v01_M",
    "ZWE_2019_MICS6_Datasets",
    "Zimbabwe MICS6 SPSS Datasets.zip"
)

spss_path = os.path.join(extract_path, "spss")
with zipfile.ZipFile(inner_zip, "r") as z:
    z.extractall(spss_path)

sav_files = []
for root, _, files in os.walk(spss_path):
    for filename in files:
        if filename.lower().endswith(".sav"):
            sav_files.append(os.path.join(root, filename))

file_paths = {os.path.basename(path): path for path in sav_files}
fs_path = file_paths["fs.sav"]

fs_data, fs_meta = pyreadstat.read_sav(fs_path)

print("Rows:", len(fs_data))
print("Variables:", len(fs_data.columns))


## 2. Verify focal variables and coding

Before recoding, the notebook checks the original metadata and raw distributions. This is important because **structural missing values must not be treated as “No.”** Special response codes are also excluded from valid denominators.


In [ ]:
focal_vars = ["PR3", "FL6A", "FL6B", "FL10"]

for var in focal_vars:
    print("\n" + "=" * 70)
    print("VARIABLE:", var)
    print("LABEL:", fs_meta.column_names_to_labels.get(var))

    label_set = fs_meta.variable_to_label.get(var)
    print("VALUE LABELS:", fs_meta.value_labels.get(label_set, {}))

    print("RAW DISTRIBUTION:")
    print(fs_data[var].value_counts(dropna=False).sort_index())


## 3. Construct clean book-access measures

For `PR3`, values **0–10** are retained as valid reported book-count categories. The special code `99` and structural missing values are excluded. In the original coding, `10` represents **10 or more books**.

`has_child_book` is then derived only for valid PR3 observations:
- `0` = no children's/picture books
- `1` = at least one children's/picture book


In [ ]:
# Valid PR3 values only
fs_data["child_books_n"] = fs_data["PR3"].where(
    fs_data["PR3"].between(0, 10)
)

# Binary access measure; preserve missingness
fs_data["has_child_book"] = np.where(
    fs_data["child_books_n"].notna(),
    fs_data["child_books_n"].gt(0).astype(int),
    np.nan
)

valid_pr3 = fs_data["child_books_n"].notna()

book_access = (
    fs_data.loc[valid_pr3, "has_child_book"]
    .value_counts()
    .sort_index()
)

book_access_pct = (
    fs_data.loc[valid_pr3, "has_child_book"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
)

print("Valid PR3 responses:", int(valid_pr3.sum()))
print("Structural/special missing:", int((~valid_pr3).sum()))
print("\nCounts:")
print(book_access)
print("\nPercentages:")
print(book_access_pct.round(2))


## 4. Create the analysis dataset

Only the focal reading variables, derived book-access variables, and selected age variables are retained for this descriptive analysis. This keeps the GitHub workflow focused while leaving the full MICS source file untouched.


In [ ]:
keep = [
    c for c in [
        "PR3", "FL6A", "FL6B", "FL10",
        "child_books_n", "has_child_book",
        "FSAGE", "schage"
    ]
    if c in fs_data.columns
]

research_data = fs_data[keep].copy()

print("Analysis rows:", len(research_data))
print("Analysis columns:", research_data.columns.tolist())

# Optional local export:
# research_data.to_csv("Zimbabwe_MICS_2019_reading_opportunity_clean.csv", index=False)


# Final Descriptive Results

Results below use valid responses for each indicator. Structural and special missing values are excluded from the relevant denominators.

These findings identify patterns in the analysed sample only; they have **not** been adjusted using MICS survey weights or the complex survey design.


## 5. Access to Children's and Picture Books


In [ ]:
no_books_pct = book_access_pct.loc[0]
some_books_pct = book_access_pct.loc[1]

book_access_table = pd.DataFrame({
    "Category": ["No children's/picture books", "At least one book"],
    "Count": [int(book_access.loc[0]), int(book_access.loc[1])],
    "Percentage": [no_books_pct, some_books_pct]
})

display(book_access_table.round(2))

categories = ["No children's/\npicture books", "At least one\nbook"]
percentages = [no_books_pct, some_books_pct]

plt.figure(figsize=(8, 5))
bars = plt.bar(categories, percentages)
plt.title("Access to Children's or Picture Books")
plt.ylabel("Percentage of valid responses (%)")
plt.ylim(0, 100)

for bar, value in zip(bars, percentages):
    plt.text(bar.get_x() + bar.get_width()/2, value + 2,
             f"{value:.1f}%", ha="center")

plt.figtext(
    0.5, -0.03,
    f"Source: Zimbabwe MICS 2019 microdata | Valid PR3 responses: n = {valid_pr3.sum():,}",
    ha="center", fontsize=9
)
plt.tight_layout()
plt.show()


In the working analysis, **4,339** children had valid PR3 responses: **3,040 (70.06%)** had no children's or picture books, while **1,299 (29.94%)** had at least one. The code above recomputes these values directly from the microdata rather than hard-coding the percentages.


## 6. Distribution of Children's and Picture Books


In [ ]:
book_dist = (
    research_data["child_books_n"]
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
)

book_pct = book_dist / book_dist.sum() * 100

book_distribution = pd.DataFrame({
    "Count": book_dist,
    "Percentage": book_pct.round(2)
})
display(book_distribution)

plt.figure(figsize=(10, 5))
labels = [str(x) if x < 10 else "10+" for x in book_pct.index]
bars = plt.bar(labels, book_pct.values)

plt.title("Distribution of Children's or Picture Books")
plt.xlabel("Number of children's or picture books")
plt.ylabel("Percentage of valid responses (%)")

for bar, value in zip(bars, book_pct.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.7,
             f"{value:.1f}%", ha="center", fontsize=9)

plt.figtext(
    0.5, -0.03,
    f"Source: Zimbabwe MICS 2019 microdata | Valid PR3 responses: n = {book_dist.sum():,}",
    ha="center", fontsize=9
)
plt.tight_layout()
plt.show()


The distribution is strongly concentrated at zero books. This is a descriptive pattern in the valid analysed sample and should not be interpreted as a weighted national estimate.


## 7. Reading Opportunity and Engagement

For `FL6A`, `FL6B`, and `FL10`, only responses coded **1 = Yes** and **2 = No** are included in the denominator. Code `9` and structural missing values are excluded.


In [ ]:
reading_vars = {
    "Child reads books": "FL6A",
    "Someone reads to child": "FL6B",
    "Child likes reading stories": "FL10"
}

results = []

for label, var in reading_vars.items():
    valid = fs_data[var].isin([1, 2])

    n_valid = int(valid.sum())
    yes_n = int((fs_data.loc[valid, var] == 1).sum())
    no_n = int((fs_data.loc[valid, var] == 2).sum())

    results.append({
        "Indicator": label,
        "Valid N": n_valid,
        "Yes N": yes_n,
        "No N": no_n,
        "Yes (%)": yes_n / n_valid * 100,
        "No (%)": no_n / n_valid * 100
    })

reading_summary = pd.DataFrame(results)
display(reading_summary.round(2))

plt.figure(figsize=(10, 5))
x = np.arange(len(reading_summary))
width = 0.36

yes_bars = plt.bar(x - width/2, reading_summary["Yes (%)"], width, label="Yes")
no_bars = plt.bar(x + width/2, reading_summary["No (%)"], width, label="No")

plt.xticks(x, reading_summary["Indicator"], rotation=0)
plt.ylabel("Percentage of valid responses (%)")
plt.title("Reading Opportunity and Engagement")
plt.ylim(0, 105)
plt.legend()

for bars in (yes_bars, no_bars):
    for bar in bars:
        value = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, value + 1,
                 f"{value:.1f}%", ha="center", fontsize=9)

plt.tight_layout()
plt.show()


In the working analysis, the valid-response results were:

- **Child reads books:** n = 4,209; 65.95% Yes, 34.05% No.
- **Someone reads books to child:** n = 4,206; 39.71% Yes, 60.29% No.
- **Child likes reading stories:** n = 4,038; 96.06% Yes, 3.94% No.

The very high reported liking of stories appears alongside more constrained material and adult-supported reading opportunities. This is best treated as an **aggregate descriptive contrast**. Because the indicators have different valid denominators, it does not establish an individual-level interest–opportunity mismatch.


## 8. Interpretation and limitations

The analysis distinguishes three related but non-equivalent dimensions:

1. **Material access** — availability of children's/picture books.
2. **Reading opportunity and practice** — whether children read books and whether someone reads to them.
3. **Reported interest** — whether the child likes reading stories.

The results are descriptive and do not establish causal relationships. They also do not show that book access, adult-supported reading, or liking stories occur together for the same individual children.

A future population-level analysis would need to incorporate the appropriate **MICS survey weights and complex survey design**. The present analysis is used instead as descriptive quantitative context for the wider research project on access to reading materials and meaningful reading opportunity.
